# Strategy comparison (CV benchmark)

In [1]:
import re
from pathlib import Path

import pandas as pd
from IPython.display import display

# ==================== Parameters ====================
# Folder with the per-instance CSV files (subfolders are searched too)
EXPERIMENTS_FOLDER = Path("../experiments")

# Datasets to include (e.g. ["cvs"], ["cvs", "g1"])
DATASETS = ["cvs"]

# Solvers to include, in this order
SOLVERS = ["ModelSolver", "BSGModelSolver", "BSGCostPredictorSolver"]

# Header label for solvers without a beam width w
SOLVER_LABELS = {"ModelSolver": "GreedyModel"}
# ====================================================

# File name: <Solver>[_w<W>]_<dataset>_<T>-<S>.csv  (e.g. BSGModelSolver_w32_cvs_3-3.csv)
PATTERN = re.compile(r"^(?P<solver>[A-Za-z]+)(?:_w(?P<w>\d+))?_(?P<dataset>[A-Za-z0-9]+)_(?P<config>\d+-\d+)\.csv$")


def rounded(table):
    """Moves with 2 decimals and time with 3, as in the paper tables."""
    out = table.copy()
    for col in out.columns:
        out[col] = out[col].round(2 if col[1] == "Moves" else 3)
    return out


def config_key(c):
    return tuple(int(x) for x in c.split("-"))


def load_results(folder):
    """Reads every CSV and returns one row per (solver, w, dataset, group)."""
    rows = []
    for path in folder.rglob("*.csv"):
        m = PATTERN.match(path.name)
        if not m:
            continue
        df = pd.read_csv(path)
        solved = df["Solved"].astype(str).str.strip().str.lower() == "true"
        rows.append({
            "solver": m["solver"],
            "w": int(m["w"]) if m["w"] else None,
            "dataset": m["dataset"],
            "Group": m["config"],
            "Solved": int(solved.sum()),
            "Total": len(df),
            "Moves": df.loc[solved, "Steps"].mean(),
            # Some CSV files (e.g. those written by g_*.py) do not record time
            "Time": df.loc[solved, "Time"].mean() if "Time" in df else float("nan"),
        })
    return pd.DataFrame(rows)


def build_table(results, solver, dataset):
    """Table with groups as rows and (w, [Moves, Time]) as columns."""
    r = results[(results["solver"] == solver) & (results["dataset"] == dataset)].copy()
    has_w = r["w"].notna().any()
    r["column"] = r["w"].astype("Int64") if has_w else SOLVER_LABELS.get(solver, solver)

    table = r.pivot(index="Group", columns="column", values=["Moves", "Time"])
    table = table.swaplevel(axis=1).sort_index(axis=1, level=0, sort_remaining=False)
    table = table.reindex(columns=["Moves", "Time"], level=1)
    table = table.reindex(sorted(table.index, key=config_key))
    table.columns.names = ["w" if has_w else None, None]

    # Warn about unsolved instances (averages are computed over solved instances only)
    unsolved = r[r["Solved"] < r["Total"]]
    for _, u in unsolved.iterrows():
        tag = f"w={u['w']:.0f} " if has_w else ""
        print(f"  ⚠ {solver} {dataset} {tag}{u['Group']}: {u['Solved']}/{u['Total']} solved")
    return table


results = load_results(EXPERIMENTS_FOLDER)
if results.empty:
    raise FileNotFoundError(f"No result CSV files found in {EXPERIMENTS_FOLDER}")

tables = {}
for dataset in DATASETS:
    for solver in SOLVERS:
        if not ((results["dataset"] == dataset) & (results["solver"] == solver)).any():
            continue
        print(f"\n=== {solver} | {dataset} ===")
        table = build_table(results, solver, dataset)
        tables[(solver, dataset)] = table
        display(rounded(table))


=== ModelSolver | cvs ===


GreedyModel       
            Moves   Time
Group                   
3-3          9.68  0.020
3-4          9.38  0.020
3-5         10.32  0.022
3-6         11.50  0.025
3-7         13.08  0.029
3-8         13.68  0.031
4-4         17.70  0.038
4-5         18.98  0.041
4-6         20.02  0.045
4-7         22.52  0.051
5-4         27.70  0.059
5-5         27.50  0.060
5-6         31.32  0.070
5-7         33.28  0.075
5-8         37.30  0.085
5-9         40.60  0.093
5-10        43.65  0.101
6-6         42.58  0.095
6-10        57.90  0.136


=== BSGModelSolver | cvs ===


w         2             4              8              16              32  \
       Moves   Time  Moves    Time  Moves    Time  Moves     Time  Moves   
Group                                                                      
3-3     8.98  0.144   8.82   0.219   8.78   0.307   8.78    0.536   8.78   
3-4     9.12  0.128   9.10   0.232   9.07   0.578   9.02    1.036   9.02   
3-5    10.22  0.152  10.18   0.278  10.15   0.804  10.15    2.370  10.15   
3-6    11.30  0.183  11.28   0.337  11.28   0.995  11.28    3.791  11.28   
3-7    12.92  0.231  12.82   0.432  12.80   1.301  12.80    5.043  12.80   
3-8    13.52  0.252  13.52   0.491  13.52   1.491  13.52    5.781  13.52   
4-4    16.45  0.389  16.02   0.691  15.92   1.734  15.88    3.145  15.85   
4-5    18.30  0.454  18.05   0.847  17.92   2.513  18.05    7.006  17.90   
4-6    19.58  0.516  19.42   0.996  19.42   3.070  19.30   11.431  19.30   
4-7    22.12  0.661  22.10   1.304  22.00   4.034  21.98   15.698  21.90   
5-4    25.05  0.888  24.30   1.656  24.08   4.067  23.90    7.457  23.85   
5-5    26.05  0.914  25.78   1.776  25.45   5.242  25.35   14.055  25.28   
5-6    30.35  1.223  30.00   2.461  29.82   7.653  29.72   27.613  29.68   
5-7    32.33  1.414  31.98   2.866  31.80   9.102  31.72   35.312  31.58   
5-8    36.33  1.801  35.92   3.754  35.83  12.062  35.65   47.067  35.58   
5-9    39.75  2.209  39.42   4.708  39.28  15.293  39.10   60.176  39.10   
5-10   42.52  2.575  42.15   5.628  41.95  18.454  41.85   72.424  41.72   
6-6    40.45  2.250  39.78   4.566  39.33  14.317  39.15   49.950  39.05   
6-10   55.88  4.620  55.28  10.328  54.98  34.421  54.75  136.313  54.62   

w               
          Time  
Group           
3-3      0.901  
3-4      1.777  
3-5      4.197  
3-6      8.866  
3-7     17.352  
3-8     22.792  
4-4      5.684  
4-5     12.770  
4-6     25.413  
4-7     49.982  
5-4     13.754  
5-5     25.955  
5-6     59.296  
5-7    105.538  
5-8    173.611  
5-9    234.412  
5-10   283.849  
6-6    105.844  
6-10   536.779


=== BSGCostPredictorSolver | cvs ===
  ⚠ BSGCostPredictorSolver cvs w=64 6-10: 39/40 solved
  ⚠ BSGCostPredictorSolver cvs w=32 6-10: 39/40 solved


w        2             4             8             16            32           \
       Moves   Time  Moves   Time  Moves   Time  Moves   Time  Moves    Time   
Group                                                                          
3-3     9.00  0.045   8.82  0.056   8.78  0.073   8.78  0.115   8.78   0.179   
3-4     9.15  0.045   9.10  0.064   9.07  0.129   9.02  0.216   9.02   0.366   
3-5    10.20  0.051  10.18  0.073  10.15  0.166  10.15  0.428  10.15   0.754   
3-6    11.30  0.057  11.28  0.082  11.28  0.190  11.28  0.630  11.28   1.395   
3-7    12.90  0.065  12.85  0.096  12.85  0.226  12.82  0.763  12.80   2.401   
3-8    13.60  0.070  13.58  0.105  13.58  0.250  13.52  0.849  13.52   3.103   
4-4    16.60  0.081  16.05  0.116  16.00  0.237  15.85  0.400  15.85   0.701   
4-5    18.52  0.092  18.08  0.134  17.98  0.310  17.95  0.759  17.92   1.376   
4-6    19.85  0.101  19.68  0.150  19.58  0.355  19.48  1.162  19.35   2.453   
4-7    22.40  0.116  22.18  0.174  22.18  0.423  22.05  1.440  21.98   4.204   
5-4    26.02  0.129  25.95  0.194  24.62  0.376  24.08  0.633  23.82   1.123   
5-5    26.55  0.134  26.08  0.200  25.82  0.470  25.50  1.105  25.38   2.020   
5-6    31.05  0.160  30.55  0.244  30.25  0.590  30.02  1.851  29.80   3.801   
5-7    33.08  0.174  32.85  0.273  32.50  0.665  32.25  2.252  32.02   6.174   
5-8    36.88  0.197  36.72  0.315  36.40  0.783  36.05  2.672  35.90   9.162   
5-9    40.35  0.220  39.98  0.355  39.67  0.902  39.62  3.094  39.30  11.397   
5-10   43.18  0.239  42.75  0.391  42.88  1.021  42.52  3.497  42.32  13.092   
6-6    41.72  0.220  41.45  0.345  40.78  0.846  40.22  2.581  39.65   5.270   
6-10   57.50  0.327  56.88  0.549  56.60  1.449  56.12  4.986  56.08  18.671   

w        64             128             256             512           
       Moves    Time  Moves     Time  Moves     Time  Moves     Time  
Group                                                                 
3-3     8.78   0.319   8.78    0.552   8.78    0.923   8.78    1.557  
3-4     9.02   0.659   9.02    1.194   9.02    2.139   9.02    4.014  
3-5    10.15   1.369  10.15    2.540  10.15    4.787  10.15    8.676  
3-6    11.28   2.538  11.28    4.746  11.28    9.034  11.28   17.373  
3-7    12.80   4.462  12.80    8.436  12.80   16.189  12.80   31.470  
3-8    13.52   6.888  13.52   13.076  13.52   25.260  13.52   49.271  
4-4    15.82   1.301  15.82    2.478  15.82    4.673  15.82    8.986  
4-5    17.88   2.550  17.88    4.951  17.85    9.556  17.85   18.056  
4-6    19.32   4.555  19.32    8.802  19.30   16.954  19.30   33.097  
4-7    21.95   7.930  21.90   15.320  21.88   29.582  21.85   57.870  
5-4    23.70   2.100  23.68    4.060  23.60    7.762  23.58   15.054  
5-5    25.30   3.772  25.22    7.350  25.12   14.199  25.08   27.080  
5-6    29.65   7.154  29.58   13.952  29.52   26.854  29.50   52.573  
5-7    31.90  11.730  31.75   22.696  31.62   44.116  31.48   86.200  
5-8    35.83  19.129  35.70   37.148  35.58   72.569  35.58  143.095  
5-9    39.25  29.384  39.20   57.672  39.08  112.917  38.85  226.093  
5-10   42.15  41.934  41.88   83.804  41.78  165.892  41.70  330.928  
6-6    39.48   9.903  39.10   19.142  38.90   37.236  38.78   72.999  
6-10   55.69  55.152  55.35  110.066  55.08  217.766  54.85  435.496

# Repeated priorities (DLTS instances, classes G1–G3)

In [2]:
import re
from pathlib import Path

import pandas as pd

EXPERIMENTS_DIR = Path("../experiments")

# File name: <Solver>[_w<W>]_<g1|g2|g3>_<T>-<S>.csv
PATTERN = re.compile(r"^(?P<solver>[A-Za-z]+)(?:_w(?P<w>\d+))?_(?P<cls>g[123])_(?P<size>\d+-\d+)\.csv$")


def summarize_csv(csv_filepath):
    df = pd.read_csv(csv_filepath)
    solved = df["Solved"].astype(str).str.strip().str.lower() == "true"
    return {
        "% Solved": solved.mean() * 100,
        "Avg. moves": df.loc[solved, "Steps"].mean(),
    }


rows = []
for f in EXPERIMENTS_DIR.glob("*.csv"):
    m = PATTERN.match(f.name)
    if not m:
        continue
    strategy = m["solver"] + (f" (w={m['w']})" if m["w"] else "")
    rows.append({
        "Class": m["cls"].upper(),
        "Size": m["size"],
        "Strategy": strategy,
        **summarize_csv(f),
    })

if not rows:
    raise FileNotFoundError(f"No g1/g2/g3 files found in {EXPERIMENTS_DIR.resolve()}")

df = pd.DataFrame(rows)

# Sort sizes numerically (5-5, 5-7, 5-10)
size_order = sorted(df["Size"].unique(), key=lambda v: tuple(int(x) for x in v.split("-")))
df["Size"] = pd.Categorical(df["Size"], categories=size_order, ordered=True)

results = (
    df.pivot_table(index=["Class", "Size"], columns="Strategy",
                   values=["% Solved", "Avg. moves"], observed=True)
    .swaplevel(axis=1)
)

# Column order of the strategies
STRATEGY_ORDER = ["ModelSolver", "BSGModelSolver", "BSGCostPredictorSolver"]
columns = sorted(
    results.columns,
    key=lambda c: (STRATEGY_ORDER.index(c[0].split(" ")[0]), c[0], c[1]),
)
results = results[columns]

results.round(2)

Strategy   ModelSolver            BSGModelSolver (w=32)             \
              % Solved Avg. moves              % Solved Avg. moves   
Class Size                                                           
G1    5-5        100.0      28.93                 100.0      26.21   
      5-7        100.0      33.80                 100.0      31.99   
      5-10       100.0      42.66                 100.0      41.01   
G2    5-5        100.0      28.26                 100.0      25.34   
      5-7        100.0      33.57                 100.0      31.39   
      5-10       100.0      42.44                 100.0      40.59   
G3    5-5        100.0      28.30                 100.0      24.60   
      5-7         98.8      33.71                 100.0      30.70   
      5-10       100.0      42.73                 100.0      40.19   

Strategy   BSGCostPredictorSolver (w=32)             
                                % Solved Avg. moves  
Class Size                                           
G1    5-5                          100.0      26.46  
      5-7                          100.0      32.31  
      5-10                         100.0      41.56  
G2    5-5                          100.0      25.46  
      5-7                          100.0      31.52  
      5-10                         100.0      40.89  
G3    5-5                          100.0      24.58  
      5-7                          100.0      30.75  
      5-10                         100.0      40.21